In [10]:
import pandas as pd
import re
import glob

### human gene name
from intervaltree import IntervalTree
import pyensembl
from Bio.Seq import Seq
from Bio import pairwise2, SeqIO
from pyensembl import EnsemblRelease
data = EnsemblRelease(release=111, species='homo_sapiens')
gene_names = data.gene_names()
gene_names = [gene for gene in gene_names if gene !='']



/home/wangp5/miniconda3/envs/vllm/lib/python3.10/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


## 1-initial filtering

In [11]:
run_id = 'run_20250501'

In [32]:
### pubmed
folder_path = "../../LLM/PubVarDB/LLM_output/LLM_output_pubmed/"  # Update this with your actual folder path
tsv_files = glob.glob(folder_path + "*.out.tsv")
df_list = [pd.read_csv(file, sep="\t",dtype=str) for file in tsv_files]
df = pd.concat(df_list, ignore_index=True)
#df=df.rename(columns={'disease':'phenotype','phenotype':'disease'})

In [33]:
df = df[df['gene'].isin(gene_names)]
df=df[(df['DNA mutation']!='-') | (df['protein mutation']!='-')]

In [34]:
### pmcid-pmid transversion
pmcid_pmid = pd.read_csv('../../LLM/pubmed/pmid_pmcid.lookup',sep='\t',header=None,dtype=str)
pmcid_pmid.columns=['PMCID','PMID']
pmcid_pmid['PMID']=pmcid_pmid['PMID'].apply(lambda x: str(x).replace('PMID:',''))

In [36]:
df_merge=df.merge(pmcid_pmid,on='PMID',how='left')
df_merge=df_merge[['PMCID','PMID', 'gene', 'DNA mutation', 'protein mutation', 'disease','phenotype',
       'LLM reasoning', 'pathogenicity']]

In [38]:
df_merge['section']='PubMed_Abstract'

In [39]:
### PMC
folder_path = "../../LLM/PubVarDB/LLM_output/LLM_output_all_pmc/"  # Update this with your actual folder path
tsv_files = glob.glob(folder_path + "*.out.tsv")

df2_list = [pd.read_csv(file, sep="\t",dtype=str) for file in tsv_files]
df2 = pd.concat(df2_list, ignore_index=True)

#df2=df2.rename(columns={'disease':'phenotype','phenotype':'disease'})
df2 = df2[df2['gene'].isin(gene_names)]
df2=df2[(df2['DNA mutation']!='-') | (df2['protein mutation']!='-')]

df2_merge = df2.merge(pmcid_pmid,on='PMCID',how='left')[['PMCID', 'PMID','section', 'gene', 'DNA mutation', 'protein mutation',
       'disease','phenotype', 'LLM reasoning', 'pathogenicity']]

### combine PMC and pubmed
df_all = pd.concat([df2_merge,df_merge]).drop_duplicates()
df_all['pathogenicity'] = df_all['pathogenicity'].apply(lambda x: str(x).strip())

OSError: Cannot save file into a non-existent directory: '../../LLM/PubVarDB/normlization/1-initial_quality_filter'

In [40]:
df_all

,PMCID,PMID,section,gene,DNA mutation,protein mutation,disease,phenotype,LLM reasoning,pathogenicity
0,PMC5770997,29375817,Results,ASPM,c.3384_3385insT,p.Lys1129Ter,-,non functional ASPM protein,induces a termination codon,pathogenic
1,PMC5771147,29343290,Results,ACKR1,rs2814778,-,malaria,malaria resistance,associated with malaria resistance,pathogenic
2,PMC5771147,29343290,Results,SLC39A4,rs1871534,-,-,zinc level,associated with zinc level,unknown
3,PMC5771147,29343290,Results,ABCC11,rs17822931,-,-,earwax and sweat type,associated with earwax and sweat type,unknown
4,PMC5771147,29343290,Results,EDAR,rs3827760,-,-,hair shape and thickness,associated with hair shape and thickness,unknown
...,...,...,...,...,...,...,...,...,...,...
503483,PMC5709726,29084757,PubMed_Abstract,CIB2,-,pTyr110*,nonsyndromic profound hearing loss,"profound hearing loss, normal balance and reti...",essential role of CIB2 in mechanotransduction ...,pathogenic
503484,PMC5649185,29085334,PubMed_Abstract,CTLA4,rs231775,-,Graves' disease,high recurrence risk,association with high recurrence risk,unknown
503485,PMC5649185,29085334,PubMed_Abstract,HLA-DRB1,-,DRB1*03,Graves' disease,high recurrence risk,association with high recurrence risk,unknown
503486,PMC5649185,29085334,PubMed_Abstract,HLA-DQA1,-,DQA1*05,Graves' disease,high recurrence risk,association with high recurrence risk,unknown


In [42]:
len(df_all['phenotype'].unique())

728149

In [43]:
len(df_all['disease'].unique())

217291

In [44]:
df_all.to_csv(f'../../LLM/PubVarDB/intermediate_DB_during_norm/1-initial_quality_filter/{run_id}.genename.tsv',sep='\t',index=False)

## 2-regex_parser

In [2]:
import re

# Dictionary to convert one-letter to three-letter amino acid codes
aa_dict = {
    'A': 'Ala', 'R': 'Arg', 'N': 'Asn', 'D': 'Asp', 'C': 'Cys',
    'Q': 'Gln', 'E': 'Glu', 'G': 'Gly', 'H': 'His', 'I': 'Ile',
    'L': 'Leu', 'K': 'Lys', 'M': 'Met', 'F': 'Phe', 'P': 'Pro',
    'S': 'Ser', 'T': 'Thr', 'W': 'Trp', 'Y': 'Tyr', 'V': 'Val',
    '*': 'Ter'  # Stop codon
    
}

def parse_protein_variant(variant):
    """Parse protein variant and return (position, from_aa, to_aa)"""
    if not isinstance(variant, str) or variant in ["-", ""]:
        return None, None, None

    # Normalize input
    variant = re.sub(r"\s+", "", variant)
    variant = variant.replace("p.", "").replace("P.", "")
    variant = variant.replace("Ter", "*").replace("stop", "*")
    variant = variant.replace("DEL", "del").replace("Del", "del").replace("INS", "ins").replace("Ins", "ins")

    patterns = [
        # 3-letter substitution + frameshift + *
        (r"([A-Za-z]{3})(\d+)([A-Za-z]{3})fs\*(\d+)", lambda m: (m[1], aa_dict.get(m[0], m[0]), f"{m[2]}fs*{m[3]}")),
        # 3-letter substitution
        (r"\b([A-Za-z]{3})(\d+)([A-Za-z]{3})\b", lambda m: (m[1], m[0], m[2])),
        # 1-letter substitution
        (r"\b([A-Z])(\d+)([A-Z])\b", lambda m: (m[1], aa_dict.get(m[0], m[0]), aa_dict.get(m[2], m[2]))),
        # frameshift + stop
        (r"([A-Za-z]{3})(\d+)fs\*(\d+)", lambda m: (m[1], aa_dict.get(m[0], m[0]), f"fs*{m[2]}")),
        (r"([A-Z])(\d+)fs\*(\d+)", lambda m: (m[1], aa_dict.get(m[0], m[0]), f"fs*{m[2]}")),
        # frameshift + no stop
        (r"([A-Za-z]{3})(\d+)fs\*", lambda m: (m[1], aa_dict.get(m[0], m[0]), "fs*")),
        (r"([A-Z])(\d+)fs\*", lambda m: (m[1], aa_dict.get(m[0], m[0]), "fs*")),
        # simple frameshift
        (r"\b([A-Za-z]{3})(\d+)fs\b", lambda m: (m[1], aa_dict.get(m[0], m[0]), "fs")),
        (r"\b([A-Z])(\d+)fs\b", lambda m: (m[1], aa_dict.get(m[0], m[0]), "fs")),
        # deletion
        (r"\b([A-Za-z]{3})(\d+)del\b", lambda m: (m[1], aa_dict.get(m[0], m[0]), "del")),
        (r"\b([A-Z])(\d+)del\b", lambda m: (m[1], aa_dict.get(m[0], m[0]), "del")),
        # termination
        (r"([A-Za-z]{3})(\d+)\*", lambda m: (m[1], aa_dict.get(m[0], m[0]), "Ter")),
        (r"([A-Z])(\d+)\*", lambda m: (m[1], aa_dict.get(m[0], m[0]), "Ter")),
        # insertion
        (r"\b(\d+)ins([A-Z])\b", lambda m: (m[0], "ins", aa_dict.get(m[1], m[1]))),
        (r"\b(\d+)ins([A-Za-z]{3})\b", lambda m: (m[0], "ins", aa_dict.get(m[1], m[1])))
    ]

    for pattern, handler in patterns:
        match = re.search(pattern, variant)
        if match:
            return handler(match.groups())

    return None, None, None

# Concise version of parse_dna_variant
def parse_dna_variant(variant):
    """Return (pos, from, to, rsid) from a DNA/cDNA mutation string."""
    if not isinstance(variant, str) or variant.strip() in ["-", ""]:
        return None, None, None, None

    variant = re.sub(r"\s+", "", variant) #removes all whitespace characters
    variant = variant.replace("→", ">").replace("c.", "").replace("C.", "").replace("g.", "").replace("G.", "")

    patterns = [
        (r"\b([\d\+]+)([ATGC]+)[-/]*[>/]([ATGC]+)\b", lambda m: (m[0], m[1], m[2], None)),
        (r"\b(\d+)([ATGC]+)[-/]*[>/]([ATGC]+)\b", lambda m: (m[0], m[1], m[2], None)),
        (r"\b([ATGC]+)(\d+)([ATGC]+)\b", lambda m: (m[1], m[0], m[2], None)),
        (r"\b(\d+)del([ATGC]*)\b", lambda m: (m[0], m[1] if m[1] else "-", "del", None)),
        (r"\bdel(\d+)([ATGC]*)\b", lambda m: (m[0], m[1] if m[1] else "-", "del", None)),
        (r"\b(\d+_\d+)del([ATGC]*)\b", lambda m: (m[0], m[1] if m[1] else "-", "del", None)),
        (r"\b([\d_]+)ins([ATGC]+)\b", lambda m: (m[0], "ins", m[1], None)),
        (r"\bins([\d_]+)([ATGC]+)\b", lambda m: (m[0], "ins", m[1], None)),
        (r"\b([\d_]+)dup([ATGC]+)\b", lambda m: (m[0], "dup", m[1], None)),
        (r"\bdup([\d_]+)([ATGC]+)\b", lambda m: (m[0], "dup", m[1], None)),
        (r"\b(rs\d+)([ATGC]+)[-/]*[>/]([ATGC]+)\b", lambda m: (None, m[1], m[2], m[0])),
        (r"\b(rs\d+)\b", lambda m: (None, None, None, m[0]))
    ]

    for pattern, handler in patterns:
        match = re.search(pattern, variant, re.IGNORECASE)
        if match:
            #turn result into upper case
            groups_upper = tuple(g.upper() if isinstance(g, str) and g.upper() in ['A','T','C','G'] else g for g in match.groups())
            return handler(groups_upper)

    return None, None, None, None



# Evaluate
###############Protein############
lst_of_test = ['p.Lys1129Ter','p.Lys1129*','P.p.Lys1129Ter',
               'p.(lys121arg)','Lys121arg','Lys121Arg','p . Asp 178 Asn',
               'A21*','T32Ter','K32stop',
               'A12K','K32N',
               'A21del','Lys32del','K3DEL',
               '21insA','32insLys','3INSK',
               'A21fs','Lys32fs',
               'A21fs*','Lys32fs*','A21fs*21','Lys32fs*31',
               '(Pro588Argfs*18)','p.Asp239Gly'
              ]

lst_of_test_results = ['Lys1129Ter','Lys1129Ter','Lys1129Ter',
               'lys121arg','Lys121arg','Lys121Arg','Asp178Asn',
               'Ala21Ter','Thr32Ter','Lys32Ter',
               'Ala12Lys','Lys32Asn',
               'Ala21del','Lys32del','Lys3del',
               'ins21Ala','ins32Lys','ins3Lys',
               'Ala21fs','Lys32fs',
               'Ala21fs*','Lys32fs*','Ala21fs*21','Lys32fs*31',
               'Pro588Argfs*18','Asp239Gly'
              ]

results = []
for i, test_str in enumerate(lst_of_test):
    parsed = parse_protein_variant(test_str)
    try:
        if not parsed[0] or not parsed[1] or not parsed[2]:
            result_str = "None"
        elif parsed[1] == "ins":
            result_str = f"ins{parsed[0]}{parsed[2]}"
        else:
            result_str = f"{parsed[1]}{parsed[0]}{parsed[2]}"
        results.append((test_str, result_str, lst_of_test_results[i],result_str == lst_of_test_results[i]))
    except Exception as e:
        results.append((test_str, f"ERROR: {e}", False))

if len([re for re in results if not re[3]])==0:
    print("Pass all tests!")
else:
    print("Failed cases:")
    print([re for re in results if not re[3]])
    

###########cDNA###########
# Example test cases and expected results
test_variants = [
    "c.123A>G", "c.(123A>G)","G123A", "123delA", "del123T",
    "c.201 + 1G > T", "c.201+1g > t",
    "123_456del", "123insA", "ins123C",
    "123dupT", "dup123G", "rs123", "RS1029524",
    "rs456C>T", "g.456T>G", "  321  A  >  T"
]
expected_outputs = [
    ("123", "A", "G", None), ("123", "A", "G", None), ("123", "G", "A", None), ("123", "A", "del", None), ("123", "T", "del", None),
    ("201+1","G","T",None),("201+1","G","T",None),
    ("123_456", "-", "del", None), ("123", "ins", "A", None), ("123", "ins", "C", None),
    ("123", "dup", "T", None), ("123", "dup", "G", None), (None, None, None, "rs123"),(None, None, None, "RS1029524"),
    (None, "C", "T", "rs456"), ("456", "T", "G", None), ("321", "A", "T", None)
]

# Evaluate function
results = []
for var, expected in zip(test_variants, expected_outputs):
    try:
        parsed = parse_dna_variant(var)
        results.append((var, parsed, expected,parsed == expected))
    except Exception as e:
        results.append((var, f"ERROR: {e}", False))

if len([re for re in results if not re[3]])==0:
    print("Pass all tests!")
else:
    print("Failed cases:")
    print([re for re in results if not re[3]])  


Pass all tests!
Pass all tests!


In [46]:
#########Extracting LLM output##########
#load
df = pd.read_csv('../../LLM/PubVarDB/intermediate_DB_during_norm/1-initial_quality_filter/run_20250501.genename.tsv',sep='\t',
                 header=0,dtype=str,low_memory=False)

### patho filter
df["pathogenicity"] = df["pathogenicity"].replace("-", "unknown")
# refine patho classification
valid_patho = ['pathogenic', 'likely pathogenic', 'conflicting', 'likely benign', 'benign', 'unknown']
df = df[df['pathogenicity'].isin(valid_patho)]

ValueError: cannot reindex on an axis with duplicate labels

In [51]:
df.columns

Index(['PMCID', 'PMID', 'section', 'gene', 'DNA mutation', 'protein mutation',
       'disease', 'phenotype', 'LLM reasoning', 'pathogenicity'],
      dtype='object')

In [53]:
df_regex = df[['gene', 'DNA mutation', 'protein mutation']].drop_duplicates()

In [54]:
df_regex

,gene,DNA mutation,protein mutation
0,ASPM,c.3384_3385insT,p.Lys1129Ter
1,ACKR1,rs2814778,-
2,SLC39A4,rs1871534,-
3,ABCC11,rs17822931,-
4,EDAR,rs3827760,-
...,...,...,...
503479,RPS10,-,p.HKYKdel
503480,RPS10,-,p.K57M
503481,RPS10,-,p.Y58F
503482,CIB2,-,pGln12*


In [55]:
df=df_regex

In [56]:
df

,gene,DNA mutation,protein mutation
0,ASPM,c.3384_3385insT,p.Lys1129Ter
1,ACKR1,rs2814778,-
2,SLC39A4,rs1871534,-
3,ABCC11,rs17822931,-
4,EDAR,rs3827760,-
...,...,...,...
503479,RPS10,-,p.HKYKdel
503480,RPS10,-,p.K57M
503481,RPS10,-,p.Y58F
503482,CIB2,-,pGln12*


In [57]:
%%time
# Apply functions to extract normalized columns
df[['dna_pos', 'dna_from', 'dna_to', 'RSID']] = df['DNA mutation'].apply(lambda x: pd.Series(parse_dna_variant(x)))
df[['aa_pos', 'aa_from', 'aa_to']] = df['protein mutation'].apply(lambda x: pd.Series(parse_protein_variant(x)))
df['dna_change']=df['dna_from']+df['dna_pos']+df['dna_to']
df['aa_change']=df['aa_from']+df['aa_pos']+df['aa_to']




CPU times: user 3min 20s, sys: 3.76 s, total: 3min 24s
Wall time: 3min 25s


In [59]:
df_all["pathogenicity"] = df_all["pathogenicity"].replace("-", "unknown")
valid_patho = ['pathogenic', 'likely pathogenic', 'conflicting', 'likely benign', 'benign', 'unknown']
df_all = df_all[df_all['pathogenicity'].isin(valid_patho)]

In [60]:
df_final = df_all.merge(df_regex,on=['gene', 'DNA mutation', 'protein mutation'],how='left')

In [61]:
df_final

,PMCID,PMID,section,gene,DNA mutation,protein mutation,disease,phenotype,LLM reasoning,pathogenicity,dna_pos,dna_from,dna_to,RSID,aa_pos,aa_from,aa_to,dna_change,aa_change
0,PMC5770997,29375817,Results,ASPM,c.3384_3385insT,p.Lys1129Ter,-,non functional ASPM protein,induces a termination codon,pathogenic,3384_3385,ins,T,None,1129,Lys,Ter,ins3384_3385T,Lys1129Ter
1,PMC5771147,29343290,Results,ACKR1,rs2814778,-,malaria,malaria resistance,associated with malaria resistance,pathogenic,None,None,None,rs2814778,None,None,None,NaN,NaN
2,PMC5771147,29343290,Results,SLC39A4,rs1871534,-,-,zinc level,associated with zinc level,unknown,None,None,None,rs1871534,None,None,None,NaN,NaN
3,PMC5771147,29343290,Results,ABCC11,rs17822931,-,-,earwax and sweat type,associated with earwax and sweat type,unknown,None,None,None,rs17822931,None,None,None,NaN,NaN
4,PMC5771147,29343290,Results,EDAR,rs3827760,-,-,hair shape and thickness,associated with hair shape and thickness,unknown,None,None,None,rs3827760,None,None,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3056013,PMC5709726,29084757,PubMed_Abstract,CIB2,-,pTyr110*,nonsyndromic profound hearing loss,"profound hearing loss, normal balance and reti...",essential role of CIB2 in mechanotransduction ...,pathogenic,None,None,None,None,110,Tyr,Ter,NaN,Tyr110Ter
3056014,PMC5649185,29085334,PubMed_Abstract,CTLA4,rs231775,-,Graves' disease,high recurrence risk,association with high recurrence risk,unknown,None,None,None,rs231775,None,None,None,NaN,NaN
3056015,PMC5649185,29085334,PubMed_Abstract,HLA-DRB1,-,DRB1*03,Graves' disease,high recurrence risk,association with high recurrence risk,unknown,None,None,None,None,1,DRB,Ter,NaN,DRB1Ter
3056016,PMC5649185,29085334,PubMed_Abstract,HLA-DQA1,-,DQA1*05,Graves' disease,high recurrence risk,association with high recurrence risk,unknown,None,None,None,None,1,DQA,Ter,NaN,DQA1Ter


In [62]:
#save
df_final.to_csv(f'../../LLM/PubVarDB/intermediate_DB_during_norm/2-variant_regex_extract/{run_id}.genename.patho_filter.variant_norm.tsv',sep='\t',index=False)

## 4-genome_coordinates

In [68]:
import pandas as pd
from pyensembl import EnsemblRelease
from Bio.Seq import Seq
import numpy as np
import logging
logging.getLogger("pyensembl.sequence_data").setLevel(logging.ERROR)


# Initialize Ensembl reference (GRCh38, release 104)
ensembl = EnsemblRelease(111, species='homo_sapiens')

aa_dict = {
    'A': 'Ala', 'R': 'Arg', 'N': 'Asn', 'D': 'Asp', 'C': 'Cys',
    'Q': 'Gln', 'E': 'Glu', 'G': 'Gly', 'H': 'His', 'I': 'Ile',
    'L': 'Leu', 'K': 'Lys', 'M': 'Met', 'F': 'Phe', 'P': 'Pro',
    'S': 'Ser', 'T': 'Thr', 'W': 'Trp', 'Y': 'Tyr', 'V': 'Val',
    '*': 'Ter'  # Stop codon
}

def get_cds_genomic_positions(tx):
    ranges = tx.coding_sequence_position_ranges
    genomic_positions = []
    for start, end in ranges:
        if tx.strand == "+":
            genomic_positions.extend(range(start, end + 1))
        else:
            # Reverse direction for minus strand
            genomic_positions.extend(range(end, start - 1, -1))
    return genomic_positions


def protein_to_genomic(gene_name, aa_pos, aa_ref=None):
    try:
        #print(len(ensembl.genes_by_name(gene_name)))
        genes = ensembl.genes_by_name(gene_name)
    except ValueError:
        #print(f'gene {gene_name} not found')
        return None  # Gene not found
    
    all_match_tx = []
    for gene in genes:
        for tx in gene.transcripts:
            if tx.is_protein_coding:
                try:
                    cds = tx.coding_sequence
                    aa_seq = Seq(cds).translate()

                    if aa_pos <= len(aa_seq):
                        aa_from_aa_seq = aa_dict.get(aa_seq[aa_pos - 1])
                        if aa_ref:
                            
                            #check if the user provided aa same as ref aa based on pos
                            if aa_ref!=aa_from_aa_seq:
                                #print('ref aa not match')
                                continue
                                
                        codon_start = (aa_pos - 1) * 3

                        # Get full CDS genomic position map
                        cds_genome = get_cds_genomic_positions(tx)

                        # get genome coodirnate based on codon_start
                        codon_coords = cds_genome[codon_start:codon_start+3]
                        # get cDNA codon
                        codon = cds[codon_start:codon_start + 3]
                        # get genome ref
                        if tx.strand=='+':
                            genome_codon = codon
                        else:
                            genome_codon = str(Seq(codon).reverse_complement())

                        all_match_tx.append({
                            "transcript_id": tx.transcript_id,
                            "chromosome": tx.contig,
                            "strand": tx.strand,
                            "genomic_coords": codon_coords,
                            "ref_aa": aa_seq[aa_pos - 1],
                            "codon_cDNA": codon,
                            'gnomic_codon': genome_codon
                        })
                        
                    #else:
                        #print('ref tx is too short')
                except Exception:
                    continue
    if len(all_match_tx) == 0:
        return None
    return all_match_tx


def cdna_to_genomic(gene_name, cdna_pos, ref_nt=None):
    try:
        genes = ensembl.genes_by_name(gene_name)
    except ValueError:
        #print(f'Gene {gene_name} not found')
        return None

    matches = []

    for gene in genes:
        for tx in gene.transcripts:
            if tx.is_protein_coding:
                try:
                    # Get spliced transcript (cDNA sequence)
                    cdna_seq = tx.coding_sequence
                    if cdna_pos > len(cdna_seq):
                        continue
                    
                    nt_from_tx = cdna_seq[cdna_pos - 1]  # 1-based to 0-based
                    
                    # Optional: check ref nucleotide match
                    if ref_nt:
                        if nt_from_tx.upper() != ref_nt.upper():
                            continue

                    # Use spliced offset → genomic mapping
                    cds_genome = get_cds_genomic_positions(tx)
                    
                    genome_coord = cds_genome[cdna_pos - 1]
                    
                    # get genome ref
                    if tx.strand=='+':
                        genome_ref = nt_from_tx
                    else:
                        genome_ref = str(Seq(nt_from_tx).complement())
                        
                    matches.append({
                        "transcript_id": tx.transcript_id,
                        "chromosome": tx.contig,
                        "strand": tx.strand,
                        "genomic_coord": genome_coord,
                        "ref_cDNA": nt_from_tx,
                        "genomic_ref": genome_ref
                    })
                except Exception as e:
                    continue
    if len(matches) == 0:
        return None
    return matches

def annotate_row(row):
    gene = row['gene']

    if gene=='-' or gene=='nan' or gene==np.nan:
        return None
    
    # Use cDNA info first
    if pd.notnull(row.get('dna_pos')) and pd.notnull(row.get('dna_from')):
        try:
            cdna_coord = cdna_to_genomic(gene, int(row['dna_pos']), row['dna_from'])
            if cdna_coord!=None:
                return cdna_coord
           
        except Exception as e:
            return f'cdna_error: {e}'
        
    # Use protein info if available
    elif pd.notnull(row.get('aa_pos')) and pd.notnull(row.get('aa_from')):
        try:
            return protein_to_genomic(gene, int(row['aa_pos']), row['aa_from'])
        except Exception as e:
            return f'protein_error: {e}'
    
    
    
    return None




In [65]:
df_final.columns

Index(['PMCID', 'PMID', 'section', 'gene', 'DNA mutation', 'protein mutation',
       'disease', 'phenotype', 'LLM reasoning', 'pathogenicity', 'dna_pos',
       'dna_from', 'dna_to', 'RSID', 'aa_pos', 'aa_from', 'aa_to',
       'dna_change', 'aa_change'],
      dtype='object')

In [66]:
df_genome_coord = df_final[['gene', 'DNA mutation', 'protein mutation',
                           'dna_pos','dna_from', 'dna_to', 'RSID', 
                            'aa_pos', 'aa_from', 'aa_to',
                   'dna_change', 'aa_change']].drop_duplicates()

In [67]:
df_genome_coord

,gene,DNA mutation,protein mutation,dna_pos,dna_from,dna_to,RSID,aa_pos,aa_from,aa_to,dna_change,aa_change
0,ASPM,c.3384_3385insT,p.Lys1129Ter,3384_3385,ins,T,None,1129,Lys,Ter,ins3384_3385T,Lys1129Ter
1,ACKR1,rs2814778,-,None,None,None,rs2814778,None,None,None,NaN,NaN
2,SLC39A4,rs1871534,-,None,None,None,rs1871534,None,None,None,NaN,NaN
3,ABCC11,rs17822931,-,None,None,None,rs17822931,None,None,None,NaN,NaN
4,EDAR,rs3827760,-,None,None,None,rs3827760,None,None,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3056009,RPS10,-,p.HKYKdel,None,None,None,None,None,None,None,NaN,NaN
3056010,RPS10,-,p.K57M,None,None,None,None,57,Lys,Met,NaN,Lys57Met
3056011,RPS10,-,p.Y58F,None,None,None,None,58,Tyr,Phe,NaN,Tyr58Phe
3056012,CIB2,-,pGln12*,None,None,None,None,12,Gln,Ter,NaN,Gln12Ter


In [69]:
%%time
df_genome_coord['genomic_coord_result'] = df_genome_coord.apply(annotate_row,axis=1)

/home/wangp5/miniconda3/envs/vllm/lib/python3.10/site-packages/Bio/Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


CPU times: user 13min 4s, sys: 2min 47s, total: 15min 52s
Wall time: 1h 1min 20s


In [70]:
df_genome_coord

,gene,DNA mutation,protein mutation,dna_pos,dna_from,dna_to,RSID,aa_pos,aa_from,aa_to,dna_change,aa_change,genomic_coord_result
0,ASPM,c.3384_3385insT,p.Lys1129Ter,3384_3385,ins,T,None,1129,Lys,Ter,ins3384_3385T,Lys1129Ter,None
1,ACKR1,rs2814778,-,None,None,None,rs2814778,None,None,None,NaN,NaN,None
2,SLC39A4,rs1871534,-,None,None,None,rs1871534,None,None,None,NaN,NaN,None
3,ABCC11,rs17822931,-,None,None,None,rs17822931,None,None,None,NaN,NaN,None
4,EDAR,rs3827760,-,None,None,None,rs3827760,None,None,None,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3056009,RPS10,-,p.HKYKdel,None,None,None,None,None,None,None,NaN,NaN,None
3056010,RPS10,-,p.K57M,None,None,None,None,57,Lys,Met,NaN,Lys57Met,None
3056011,RPS10,-,p.Y58F,None,None,None,None,58,Tyr,Phe,NaN,Tyr58Phe,None
3056012,CIB2,-,pGln12*,None,None,None,None,12,Gln,Ter,NaN,Gln12Ter,"[{'transcript_id': 'ENST00000258930', 'chromos..."


In [87]:
import pandas as pd
import ast
from Bio.Seq import Seq
from Bio.Data import CodonTable

# 3-letter to 1-letter amino acid dictionary
aa_dict = {
    'A': 'Ala', 'R': 'Arg', 'N': 'Asn', 'D': 'Asp', 'C': 'Cys',
    'Q': 'Gln', 'E': 'Glu', 'G': 'Gly', 'H': 'His', 'I': 'Ile',
    'L': 'Leu', 'K': 'Lys', 'M': 'Met', 'F': 'Phe', 'P': 'Pro',
    'S': 'Ser', 'T': 'Thr', 'W': 'Trp', 'Y': 'Tyr', 'V': 'Val',
    '*': 'Ter'
}
aa_3to1 = {v: k for k, v in aa_dict.items()}

# Amino acid to RNA codon table
standard_table = CodonTable.unambiguous_rna_by_name["Standard"]
aa_to_codons = {}
for codon, aa in standard_table.forward_table.items():
    aa_to_codons.setdefault(aa, []).append(codon)
for codon in standard_table.stop_codons:
    aa_to_codons.setdefault("*", []).append(codon)


def get_cdna_variant(tx, dna_pos, dna_from, dna_to):
    if 'ref_cDNA' not in tx or 'genomic_coord' not in tx:
        return None

    # Compare reference base at cDNA level
    if str(tx['ref_cDNA']).upper() != str(dna_from).upper():
        return None

    # genomic_coord is a single position (int), not a list
    pos = tx['genomic_coord']
    ref = str(dna_from).upper()
    alt = str(dna_to).upper()

    # Adjust for strand
    if tx['strand'] == '-':
        ref = str(Seq(ref).complement())
        alt = str(Seq(alt).complement())

    return {
        'chr': tx['chromosome'],
        'pos': pos,
        'ref': ref,
        'alt': alt,
        'transcript_id': tx['transcript_id']
    }

    
    
def get_protein_variant(tx, aa_pos, aa_from, aa_to):
    if 'ref_aa' not in tx or 'genomic_coords' not in tx or 'gnomic_codon' not in tx:
        return []

    # Convert 3-letter to 1-letter amino acid
    aa_from_1 = aa_3to1.get(str(aa_from).capitalize())
    aa_to_1 = aa_3to1.get(str(aa_to).capitalize())
    if not aa_from_1 or not aa_to_1:
        return []

    if tx['ref_aa'].upper() != aa_from_1:
        return []

    ref_codon = tx['gnomic_codon'].upper()
    strand = tx['strand']
    coords = tx['genomic_coords']
    variants = []

    for alt_rna_codon in aa_to_codons.get(aa_to_1, []):
        # Convert RNA to DNA
        alt_dna_codon = alt_rna_codon.replace("U", "T")

        if strand == '+':
            alt_genomic_codon = alt_dna_codon
            start_coord = coords[0]
        else:
            alt_genomic_codon = str(Seq(alt_dna_codon).reverse_complement())
            start_coord = coords[-1]

        if ref_codon != alt_genomic_codon:
            variants.append({
                'chr': tx['chromosome'],
                'pos': start_coord,  # always the leftmost genomic position
                'ref': ref_codon,
                'alt': alt_genomic_codon,
                'transcript_id': tx['transcript_id'],
                'aa_from': aa_from,
                'aa_to': aa_to
            })

    return variants


def parse_variant_from_transcript(row, mane_tx_dict):
    gene = row['gene']
    results = []
    if row['genomic_coord_result']==None:
        return results
    
    try:
        if isinstance(row['genomic_coord_result'], str):
            if row['genomic_coord_result'].strip().startswith("[{"):
                transcripts = ast.literal_eval(row['genomic_coord_result'])
            else:
                return []  # or None
        else:
            transcripts = row['genomic_coord_result']
    except Exception:
        return []  # or log the row and skip
        
    # Extract variant information
    dna_pos = row.get('dna_pos')
    dna_from = row.get('dna_from')
    dna_to   = row.get('dna_to')
    aa_pos   = row.get('aa_pos')
    aa_from  = row.get('aa_from')
    aa_to    = row.get('aa_to')

    mane_tx = mane_tx_dict.get(gene)
    

    for tx in transcripts:
        is_mane = (tx['transcript_id'] == mane_tx)

        if 'ref_aa' in tx and pd.notnull(aa_pos) and pd.notnull(aa_from) and pd.notnull(aa_to):
            variants = get_protein_variant(tx, aa_pos, aa_from, aa_to)
        elif 'ref_cDNA' in tx and pd.notnull(dna_pos) and pd.notnull(dna_from) and pd.notnull(dna_to):
            variant = get_cdna_variant(tx, dna_pos, dna_from, dna_to)
            variants = [variant] if variant else []
        else:
            continue

        for var in variants:
            if var:
                var_clean = {
                    'chr': var['chr'],
                    'pos': var['pos'],
                    'ref': var['ref'],
                    'alt': var['alt'],
                    'MANE_transcript_used': str(is_mane)
                }
                results.append(var_clean)

    # Deduplicate by chr, pos, ref, alt
    seen = set()
    unique_results = []
    for r in results:
        key = (r['chr'], r['pos'], r['ref'], r['alt'])
        if key not in seen:
            seen.add(key)
            unique_results.append(r)

    return unique_results if unique_results else None



In [74]:
mane_select = pd.read_csv('/mnt/isilon/wang_lab/pengwang/projects/LLM/variant_coordinate/mane.bed',sep='\t',header=None)
mane_select['ENST']=mane_select[3].apply(lambda x: x.split('.')[0])
mane_tx_df = mane_select[[18,'ENST']].rename(columns={18:'gene'})
mane_tx_dict = dict(zip(mane_tx_df['gene'], mane_tx_df['ENST']))

In [89]:
%%time
df_genome_coord['parsed_variants'] = df_genome_coord.apply(lambda row: 
                                            parse_variant_from_transcript(row, mane_tx_dict), axis=1)


CPU times: user 25 s, sys: 446 ms, total: 25.4 s
Wall time: 25.6 s


In [90]:
df_genome_coord

,gene,DNA mutation,protein mutation,dna_pos,dna_from,dna_to,RSID,aa_pos,aa_from,aa_to,dna_change,aa_change,genomic_coord_result,parsed_variants
0,ASPM,c.3384_3385insT,p.Lys1129Ter,3384_3385,ins,T,None,1129,Lys,Ter,ins3384_3385T,Lys1129Ter,None,[]
1,ACKR1,rs2814778,-,None,None,None,rs2814778,None,None,None,NaN,NaN,None,[]
2,SLC39A4,rs1871534,-,None,None,None,rs1871534,None,None,None,NaN,NaN,None,[]
3,ABCC11,rs17822931,-,None,None,None,rs17822931,None,None,None,NaN,NaN,None,[]
4,EDAR,rs3827760,-,None,None,None,rs3827760,None,None,None,NaN,NaN,None,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3056009,RPS10,-,p.HKYKdel,None,None,None,None,None,None,None,NaN,NaN,None,[]
3056010,RPS10,-,p.K57M,None,None,None,None,57,Lys,Met,NaN,Lys57Met,None,[]
3056011,RPS10,-,p.Y58F,None,None,None,None,58,Tyr,Phe,NaN,Tyr58Phe,None,[]
3056012,CIB2,-,pGln12*,None,None,None,None,12,Gln,Ter,NaN,Gln12Ter,"[{'transcript_id': 'ENST00000258930', 'chromos...","[{'chr': '15', 'pos': 78131180, 'ref': 'CTG', ..."


In [92]:
df_genome_coord_final = df_final.merge(df_genome_coord,on=['gene', 'DNA mutation', 'protein mutation',
                           'dna_pos','dna_from', 'dna_to', 'RSID', 
                            'aa_pos', 'aa_from', 'aa_to',
                   'dna_change', 'aa_change'],how='left')

In [93]:
df_genome_coord_final.to_csv(f'../../LLM/PubVarDB/intermediate_DB_during_norm/4-genome_coordinates/{run_id}.genename.patho_filter.variant_norm.ensembl111.tsv',sep='\t',index=False)